# 10 — Factuality Grounding

Entity linking and knowledge graph verification anchor LLM claims to verifiable facts.

## Entity Linking and Knowledge Graph Verification

**Factuality grounding** connects LLM claims to structured knowledge sources:

1. **Entity extraction**: identify named entities (people, places, dates, quantities) in the response
2. **Entity linking**: map extracted entities to knowledge graph (KG) nodes (Wikidata, DBpedia)
3. **Relation verification**: check if the stated relationship between entities is in the KG
4. **Score**: fraction of verifiable claims confirmed by the KG

**FActScore approach** (Min et al., 2023): decompose a long-form biography into atomic facts, then verify each against a Wikipedia-derived KG. This gives a per-claim precision score for LLM outputs.

**Limitations of KG grounding**:
- KGs have limited coverage: niche facts, recent events, and informal knowledge are absent
- Temporal gaps: KGs may have outdated information
- Ambiguity: entity resolution is hard for common names

**Practical recipe for production**: use retrieval (Google, Wikipedia API) rather than a static KG to verify claims about recent events.

In [ ]:
# Factuality pipeline: claim -> KG verification
import re

# Simulated knowledge graph
KG = {
    ('Paris', 'capital_of'): 'France',
    ('Paris', 'founded_by'): 'Romans',
    ('Eiffel Tower', 'location'): 'Paris',
    ('Eiffel Tower', 'completed'): '1889',
    ('Eiffel Tower', 'designed_by'): 'Gustave Eiffel',
    ('France', 'population'): '68 million',
    ('France', 'official_language'): 'French',
    ('Python', 'created_by'): 'Guido van Rossum',
    ('Python', 'first_release'): '1991',
    ('Einstein', 'known_for'): 'Theory of relativity',
    ('Einstein', 'nobel_prize'): '1921',
}

def extract_claims(text):
    """Extract simple subject-predicate-object triples (simplified)."""
    patterns = [
        (r'(\w+(?:\s\w+)?) is (?:the )?capital of (\w+)', 'capital_of'),
        (r'(\w+(?:\s\w+)?) was (?:built|completed) in (\d{4})', 'completed'),
        (r'(\w+(?:\s\w+)?) is located in (\w+)', 'location'),
        (r'(\w+(?:\s\w+)?) was created by (\w+(?:\s\w+)?)', 'created_by'),
        (r'(\w+(?:\s\w+)?) has (?:a )?population of ([\d\s]+million)', 'population'),
    ]

    claims = []
    for pattern, relation in patterns:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            subject = match.group(1).strip().title()
            obj = match.group(2).strip()
            claims.append({'subject': subject, 'relation': relation, 'object': obj})
    return claims

def verify_claim(claim, kg):
    """Check if claim is in KG. Returns True/False/None (None=can't verify)."""
    key = (claim['subject'], claim['relation'])
    if key in kg:
        kg_val = kg[key].lower()
        claim_val = claim['object'].lower()
        # Fuzzy match
        return claim_val in kg_val or kg_val in claim_val
    return None  # not in KG

# Test texts
texts = [
    'Paris is the capital of France. The Eiffel Tower was completed in 1889. France has a population of 68 million.',
    'Paris is the capital of Belgium. The Eiffel Tower was completed in 1850. France has a population of 100 million.',
    'Python was created by Guido van Rossum. Python was completed in 1985.',
]

for text in texts:
    claims = extract_claims(text)
    verified = [verify_claim(c, KG) for c in claims]
    n_verified = sum(1 for v in verified if v is not None)
    n_correct = sum(1 for v in verified if v is True)
    factscore = n_correct / max(n_verified, 1)

    print(f'Text: "{text[:60]}..."')
    print(f'FActScore: {factscore:.2f} ({n_correct}/{n_verified} claims verified)')
    for c, v in zip(claims, verified):
        status = 'CORRECT' if v is True else ('WRONG' if v is False else 'NOT IN KG')
        print(f'  [{status}] {c["subject"]} -[{c["relation"]}]-> {c["object"]}')
    print()